# Stacking Ensemble — Credit Risk Default Prediction

**Stacked generalization:** multiple diverse base models → OOF predictions via K-Fold CV → meta-model trained on OOF features.

### Base models (4 diverse learners)
- **XGBoost** — gradient boosting
- **Random Forest** — bagging
- **MLP-A** — [64, 32, 16] + BatchNorm + Dropout
- **MLP-B** — [128, 64] + Dropout (different inductive bias)

### Meta-model
Logistic Regression — simple, calibrated, resists overfitting on 4 meta-features.

### Evaluation
Compare: individual base models · simple averaging · stacking ensemble

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import pickle
import os
from copy import deepcopy

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    roc_auc_score, classification_report, confusion_matrix,
    precision_score, recall_score, f1_score, brier_score_loss,
    precision_recall_curve, average_precision_score, RocCurveDisplay
)
from xgboost import XGBClassifier

sns.set_theme(style="whitegrid")
plt.rcParams["figure.dpi"] = 120

torch.manual_seed(42)
np.random.seed(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
print(f"PyTorch: {torch.__version__}  |  XGBoost: {XGBClassifier.__module__}")

Device: cpu
PyTorch: 2.12.0+cu130  |  XGBoost: xgboost.sklearn


/home/momo/Documents/internship/week-03/credit-risk-pipeline/.venv/lib/python3.12/site-packages/torch/cuda/__init__.py:187: UserWarning: CUDA initialization: The NVIDIA driver on your system is too old (found version 12020). Please update your GPU driver by downloading and installing a new version from the URL: http://www.nvidia.com/Download/index.aspx Alternatively, go to: https://pytorch.org to install a PyTorch version that has been compiled with your version of the CUDA driver. (Triggered internally at /pytorch/c10/cuda/CUDAFunctions.cpp:119.)
  return torch._C._cuda_getDeviceCount() > 0


In [2]:
# =============================================================================
# Environment setup — works both locally and in Google Colab
# =============================================================================
# In Colab: upload train.csv, val.csv, test.csv via the file browser,
# or mount Google Drive and set DATA_PATH below.
# Locally: reads from ../../data/processed/

try:
    import google.colab
    IN_COLAB = True
    print("Google Colab detected.")
    print("Upload train.csv, val.csv, test.csv to /content/ or set DATA_PATH below.")
    DATA_PATH = "/content"  # change to your Drive path if mounting
except ImportError:
    IN_COLAB = False
    DATA_PATH = "../../data/processed"
    print("Local environment detected.")

# Output directory — models & plots saved here
OUT_DIR = "./output"
os.makedirs(OUT_DIR, exist_ok=True)
print(f"DATA_PATH: {DATA_PATH}")
print(f"OUT_DIR:   {OUT_DIR}")

Local environment detected.
DATA_PATH: ../../data/processed
OUT_DIR:   ./output


In [3]:
# --- Load data ---
# Reads from DATA_PATH (set in the environment setup cell above)
train_df = pd.read_csv(f"{DATA_PATH}/train.csv")
val_df   = pd.read_csv(f"{DATA_PATH}/val.csv")
test_df  = pd.read_csv(f"{DATA_PATH}/test.csv")

# Combine train + val for K-Fold CV (val is used for early stopping in NN base models)
full_train_df = pd.concat([train_df, val_df], ignore_index=True)

X_full = full_train_df.drop(columns=['DEFAULT_OCT']).values.astype(np.float32)
y_full = full_train_df['DEFAULT_OCT'].values.astype(np.float32)

X_test = test_df.drop(columns=['DEFAULT_OCT']).values.astype(np.float32)
y_test = test_df['DEFAULT_OCT'].values.astype(np.float32)

print(f"Full train (for CV): {X_full.shape[0]:,} samples, {X_full.shape[1]} features")
print(f"Test:               {X_test.shape[0]:,} samples")
print(f"Default rate: {y_full.mean():.4f}")

Full train (for CV): 25,500 samples, 10 features
Test:               4,500 samples
Default rate: 0.2212


In [4]:
# =============================================================================
# PyTorch MLP definitions
# =============================================================================

class MLPBase(nn.Module):
    """Generic MLP with configurable hidden dims and dropout."""
    def __init__(self, input_dim, hidden_dims, dropout_rates, use_batchnorm=True):
        super().__init__()
        layers = []
        prev = input_dim
        for hd, dr in zip(hidden_dims, dropout_rates):
            layers.append(nn.Linear(prev, hd))
            if use_batchnorm:
                layers.append(nn.BatchNorm1d(hd))
            layers.append(nn.ReLU())
            layers.append(nn.Dropout(dr))
            prev = hd
        self.backbone = nn.Sequential(*layers)
        self.head = nn.Linear(prev, 1)

    def forward(self, x):
        return self.head(self.backbone(x)).squeeze(-1)


def make_mlp_a(input_dim):
    """MLP-A: [64, 32, 16] + BatchNorm + Dropout"""
    return MLPBase(input_dim, [64, 32, 16], [0.3, 0.3, 0.2], use_batchnorm=True)


def make_mlp_b(input_dim):
    """MLP-B: [128, 64] + Dropout only (no BatchNorm) — different inductive bias"""
    return MLPBase(input_dim, [128, 64], [0.4, 0.3], use_batchnorm=False)

input_dim = X_full.shape[1]
print(f"Input dimension: {input_dim}")

Input dimension: 10


In [5]:
# =============================================================================
# PyTorch training helper
# =============================================================================

def train_mlp(model, X_train, y_train, X_val, y_val,
              epochs=80, batch_size=256, lr=1e-3, patience=15, verbose=False):
    """Train an MLP with early stopping. Returns trained model."""
    model = model.to(device)

    pos_count = y_train.sum()
    neg_count = len(y_train) - pos_count
    pos_weight = neg_count / max(pos_count, 1)

    criterion = nn.BCEWithLogitsLoss(pos_weight=torch.tensor(pos_weight))
    optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=1e-5)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='min', factor=0.5, patience=8
    )

    train_ds = TensorDataset(
        torch.tensor(X_train, dtype=torch.float32),
        torch.tensor(y_train, dtype=torch.float32)
    )
    val_ds = TensorDataset(
        torch.tensor(X_val, dtype=torch.float32),
        torch.tensor(y_val, dtype=torch.float32)
    )

    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
    val_loader   = DataLoader(val_ds,   batch_size=batch_size * 2, shuffle=False)

    best_val_loss = float('inf')
    best_state = None
    counter = 0

    for epoch in range(1, epochs + 1):
        model.train()
        train_loss = 0.0
        for Xb, yb in train_loader:
            Xb, yb = Xb.to(device), yb.to(device)
            optimizer.zero_grad()
            loss = criterion(model(Xb), yb)
            loss.backward()
            optimizer.step()
            train_loss += loss.item() * Xb.size(0)
        train_loss /= len(train_ds)

        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for Xb, yb in val_loader:
                Xb, yb = Xb.to(device), yb.to(device)
                val_loss += criterion(model(Xb), yb).item() * Xb.size(0)
        val_loss /= len(val_ds)

        scheduler.step(val_loss)

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_state = deepcopy(model.state_dict())
            counter = 0
        else:
            counter += 1

        if counter >= patience:
            break

    model.load_state_dict(best_state)
    return model


def predict_mlp(model, X):
    """Return probability predictions for an MLP."""
    model.eval()
    model = model.to(device)
    ds = TensorDataset(torch.tensor(X, dtype=torch.float32))
    loader = DataLoader(ds, batch_size=1024, shuffle=False)
    probs = []
    with torch.no_grad():
        for (Xb,) in loader:
            Xb = Xb.to(device)
            probs.extend(torch.sigmoid(model(Xb)).cpu().numpy())
    return np.array(probs)

In [6]:
# =============================================================================
# Optuna Tuning — XGBoost, RandomForest, Meta-model C
# =============================================================================
RUN_TUNING = True
N_TRIALS   = 25

if RUN_TUNING:
    import optuna
    from optuna.samplers import TPESampler

    # Single train/val split for fast tuning (not the full K-fold)
    X_tune, X_vtune, y_tune, y_vtune = train_test_split(
        X_full, y_full, test_size=0.20, stratify=y_full, random_state=42
    )

    def objective(trial):
        # --- 1. XGBoost ---
        xgb = XGBClassifier(
            n_estimators=trial.suggest_int('xgb_n_estimators', 100, 400),
            max_depth=trial.suggest_int('xgb_max_depth', 3, 10),
            learning_rate=trial.suggest_float('xgb_lr', 0.01, 0.2, log=True),
            subsample=trial.suggest_float('xgb_subsample', 0.6, 1.0),
            colsample_bytree=trial.suggest_float('xgb_colsample', 0.6, 1.0),
            scale_pos_weight=(len(y_tune)-y_tune.sum())/y_tune.sum(),
            random_state=42, eval_metric='logloss', verbosity=0
        )
        xgb.fit(X_tune, y_tune)
        xgb_auc = roc_auc_score(y_vtune, xgb.predict_proba(X_vtune)[:, 1])

        # --- 2. Random Forest ---
        rf = RandomForestClassifier(
            n_estimators=trial.suggest_int('rf_n_estimators', 100, 400),
            max_depth=trial.suggest_int('rf_max_depth', 6, 20),
            min_samples_leaf=trial.suggest_int('rf_min_leaf', 5, 50),
            class_weight='balanced', random_state=42, n_jobs=-1
        )
        rf.fit(X_tune, y_tune)
        rf_auc = roc_auc_score(y_vtune, rf.predict_proba(X_vtune)[:, 1])

        # Combined score (average — both matter)
        return (xgb_auc + rf_auc) / 2

    print(f"Tuning XGBoost + RandomForest ({N_TRIALS} trials)...\n")
    study = optuna.create_study(direction='maximize', sampler=TPESampler(seed=42))
    study.optimize(objective, n_trials=N_TRIALS, show_progress_bar=True)

    print(f"\nBest trial #{study.best_trial.number}")
    print(f"Best avg AUC: {study.best_value:.4f}")
    for k, v in study.best_params.items():
        print(f"  {k:25s} = {v}")

    BEST_TREE_PARAMS = study.best_params

    # --- Quick meta-model C tuning (on 3-fold CV to avoid re-running full pipeline) ---
    # Use default base models for a rough meta-C search
    print("\nTuning meta-model C...")
    C_values = [0.01, 0.1, 0.5, 1.0, 5.0, 10.0]
    best_c, best_c_auc = 1.0, 0.0
    for c in C_values:
        # Simple 3-fold to evaluate meta-C
        skf_small = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
        oof_c = np.zeros(X_full.shape[0])
        for tr_idx, oof_idx in skf_small.split(X_full, y_full):
            X_tr_c, y_tr_c = X_full[tr_idx], y_full[tr_idx]
            X_oof_c = X_full[oof_idx]

            # Quick base models
            xgb_c = XGBClassifier(n_estimators=100, max_depth=5, learning_rate=0.05,
                                  scale_pos_weight=(len(y_tr_c)-y_tr_c.sum())/y_tr_c.sum(),
                                  random_state=42, eval_metric='logloss', verbosity=0)
            xgb_c.fit(X_tr_c, y_tr_c)
            rf_c = RandomForestClassifier(n_estimators=100, max_depth=10, min_samples_leaf=20,
                                          class_weight='balanced', random_state=42, n_jobs=-1)
            rf_c.fit(X_tr_c, y_tr_c)
            oof_c[oof_idx] = (xgb_c.predict_proba(X_oof_c)[:,1] + rf_c.predict_proba(X_oof_c)[:,1]) / 2

        #  This is just averaging, not full stacking — sufficient for choosing C
        auc_c = roc_auc_score(y_full, oof_c)
        if auc_c > best_c_auc:
            best_c_auc, best_c = auc_c, c

    print(f"  Best meta C: {best_c} (avg-AUC proxy: {best_c_auc:.4f})")
    BEST_META_C = best_c

else:
    print("Tuning skipped. Using defaults.")
    BEST_TREE_PARAMS = {}
    BEST_META_C = 1.0

/home/momo/Documents/internship/week-03/credit-risk-pipeline/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[I 2026-06-10 16:30:47,518] A new study created in memory with name: no-name-78560ea2-b371-49f1-81af-52e9195cff81


Tuning XGBoost + RandomForest (25 trials)...



Best trial: 0. Best value: 0.767858:   4%|▍         | 1/25 [00:01<00:24,  1.04s/it]

[I 2026-06-10 16:30:48,555] Trial 0 finished with value: 0.7678580515737825 and parameters: {'xgb_n_estimators': 212, 'xgb_max_depth': 10, 'xgb_lr': 0.08960785365368121, 'xgb_subsample': 0.8394633936788146, 'xgb_colsample': 0.6624074561769746, 'rf_n_estimators': 146, 'rf_max_depth': 6, 'rf_min_leaf': 44}. Best is trial 0 with value: 0.7678580515737825.


Best trial: 1. Best value: 0.783534:   8%|▊         | 2/25 [00:01<00:22,  1.03it/s]

...

[I 2026-06-10 16:31:13,831] Trial 22 finished with value: 0.7868986942283931 and parameters: {'xgb_n_estimators': 276, 'xgb_max_depth': 5, 'xgb_lr': 0.02074153469376734, 'xgb_subsample': 0.7710130615017814, 'xgb_colsample': 0.9945925898419629, 'rf_n_estimators': 363, 'rf_max_depth': 16, 'rf_min_leaf': 50}. Best is trial 21 with value: 0.7869042182690178.


Best trial: 23. Best value: 0.786944:  96%|█████████▌| 24/25 [00:27<00:01,  1.51s/it]

[I 2026-06-10 16:31:15,516] Trial 23 finished with value: 0.7869436119324633 and parameters: {'xgb_n_estimators': 247, 'xgb_max_depth': 5, 'xgb_lr': 0.02233295506329898, 'xgb_subsample': 0.7617829604997519, 'xgb_colsample': 0.9414790499409719, 'rf_n_estimators': 391, 'rf_max_depth': 16, 'rf_min_leaf': 44}. Best is trial 23 with value: 0.7869436119324633.


Best trial: 23. Best value: 0.786944: 100%|██████████| 25/25 [00:29<00:00,  1.20s/it]


[I 2026-06-10 16:31:17,394] Trial 24 finished with value: 0.7866359396091792 and parameters: {'xgb_n_estimators': 239, 'xgb_max_depth': 6, 'xgb_lr': 0.0258352169895324, 'xgb_subsample': 0.7495869415605074, 'xgb_colsample': 0.9372147987061042, 'rf_n_estimators': 394, 'rf_max_depth': 15, 'rf_min_leaf': 44}. Best is trial 23 with value: 0.7869436119324633.

Best trial #23
Best avg AUC: 0.7869
  xgb_n_estimators          = 247
  xgb_max_depth             = 5
  xgb_lr                    = 0.02233295506329898
  xgb_subsample             = 0.7617829604997519
  xgb_colsample             = 0.9414790499409719
  rf_n_estimators           = 391
  rf_max_depth              = 16
  rf_min_leaf               = 44

Tuning meta-model C...
  Best meta C: 0.01 (avg-AUC proxy: 0.7822)


# Hyperparameter Tuning with Optuna

Tunes XGBoost + RandomForest + meta-model C. MLP architectures use the best from the single-MLP notebook tuning.

In [ ]:
# =============================================================================
# K-Fold Cross-Validation → Out-of-Fold (OOF) predictions
# =============================================================================
N_FOLDS = 5
skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=42)

oof_predictions = np.zeros((X_full.shape[0], 4))
oof_labels = np.zeros(X_full.shape[0])
base_model_names = ['XGBoost', 'RandomForest', 'MLP-A', 'MLP-B']

# Use tuned params if available
_xgb_defaults = dict(n_estimators=200, max_depth=5, learning_rate=0.05,
                     subsample=1.0, colsample_bytree=1.0)
_rf_defaults  = dict(n_estimators=200, max_depth=12, min_samples_leaf=20)

for fold_idx, (train_idx, oof_idx) in enumerate(skf.split(X_full, y_full)):
    print(f"\n{'='*60}")
    print(f"Fold {fold_idx + 1}/{N_FOLDS}")
    print(f"{'='*60}")

    X_tr, y_tr = X_full[train_idx], y_full[train_idx]
    X_oof, y_oof = X_full[oof_idx], y_full[oof_idx]

    X_nn_tr, X_nn_val, y_nn_tr, y_nn_val = train_test_split(
        X_tr, y_tr, test_size=0.15, stratify=y_tr, random_state=42 + fold_idx
    )

    # --- 1. XGBoost (tuned or default) ---
    xgb_kw = {k: BEST_TREE_PARAMS.get(f'xgb_{k}', _xgb_defaults[k])
              for k in ['n_estimators','max_depth','learning_rate','subsample','colsample_bytree']}
    xgb = XGBClassifier(**xgb_kw,
        scale_pos_weight=(len(y_tr) - y_tr.sum()) / y_tr.sum(),
        random_state=42, eval_metric='logloss', verbosity=0)
    xgb.fit(X_tr, y_tr)
    oof_predictions[oof_idx, 0] = xgb.predict_proba(X_oof)[:, 1]
    print(f"  XGBoost          — trained")

    # --- 2. Random Forest (tuned or default) ---
    rf_kw = {k: BEST_TREE_PARAMS.get(f'rf_{k}', _rf_defaults[k])
             for k in ['n_estimators','max_depth','min_samples_leaf']}
    rf = RandomForestClassifier(**rf_kw,
        class_weight='balanced', random_state=42, n_jobs=-1)
    rf.fit(X_tr, y_tr)
    oof_predictions[oof_idx, 1] = rf.predict_proba(X_oof)[:, 1]
    print(f"  RandomForest     — trained")

    # --- 3. MLP-A ---
    mlp_a = make_mlp_a(input_dim)
    mlp_a = train_mlp(mlp_a, X_nn_tr, y_nn_tr, X_nn_val, y_nn_val, verbose=False)
    oof_predictions[oof_idx, 2] = predict_mlp(mlp_a, X_oof)
    print(f"  MLP-A [64,32,16] — trained")

    # --- 4. MLP-B ---
    mlp_b = make_mlp_b(input_dim)
    mlp_b = train_mlp(mlp_b, X_nn_tr, y_nn_tr, X_nn_val, y_nn_val, verbose=False)
    oof_predictions[oof_idx, 3] = predict_mlp(mlp_b, X_oof)
    print(f"  MLP-B [128,64]   — trained")

    oof_labels[oof_idx] = y_oof

print(f"\nOOF predictions shape: {oof_predictions.shape}")

In [ ]:
# --- OOF AUC per base model (sanity check) ---
for i, name in enumerate(base_model_names):
    auc = roc_auc_score(oof_labels, oof_predictions[:, i])
    print(f"OOF AUC  {name:15s}: {auc:.4f}")

# Simple average of OOF predictions
avg_oof = oof_predictions.mean(axis=1)
print(f"{'OOF AUC  Simple Average':15s}: {roc_auc_score(oof_labels, avg_oof):.4f}")

In [ ]:
# =============================================================================
# Train meta-model on OOF predictions
# =============================================================================
meta_model = LogisticRegression(C=BEST_META_C, max_iter=1000, random_state=42)
meta_model.fit(oof_predictions, oof_labels)

coef_df = pd.DataFrame({
    'Model': base_model_names,
    'Weight': meta_model.coef_[0]
}).sort_values('Weight', ascending=False)

print(f"Meta-model (Logistic Regression, C={BEST_META_C}) coefficients:")
print(coef_df.to_string(index=False))
print(f"\nIntercept: {meta_model.intercept_[0]:.4f}")

In [ ]:
# --- OOF AUC of the stacking ensemble ---
oof_stacked_probs = meta_model.predict_proba(oof_predictions)[:, 1]
stacked_auc_oof = roc_auc_score(oof_labels, oof_stacked_probs)
print(f"OOF AUC  Stacking Ensemble: {stacked_auc_oof:.4f}")

In [ ]:
# =============================================================================
# Retrain each base model on the FULL training set
# =============================================================================
print("Retraining base models on full training set...\n")

X_nn_tr, X_nn_val, y_nn_tr, y_nn_val = train_test_split(
    X_full, y_full, test_size=0.10, stratify=y_full, random_state=42
)

# --- XGBoost (tuned or default) ---
xgb_final = XGBClassifier(**xgb_kw,
    scale_pos_weight=(len(y_full) - y_full.sum()) / y_full.sum(),
    random_state=42, eval_metric='logloss', verbosity=0)
xgb_final.fit(X_full, y_full)
print("  XGBoost          ✓")

# --- Random Forest (tuned or default) ---
rf_final = RandomForestClassifier(**rf_kw,
    class_weight='balanced', random_state=42, n_jobs=-1)
rf_final.fit(X_full, y_full)
print("  RandomForest     ✓")

# --- MLP-A ---
mlp_a_final = make_mlp_a(input_dim)
mlp_a_final = train_mlp(mlp_a_final, X_nn_tr, y_nn_tr, X_nn_val, y_nn_val, verbose=False)
print("  MLP-A [64,32,16] ✓")

# --- MLP-B ---
mlp_b_final = make_mlp_b(input_dim)
mlp_b_final = train_mlp(mlp_b_final, X_nn_tr, y_nn_tr, X_nn_val, y_nn_val, verbose=False)
print("  MLP-B [128,64]   ✓")

final_models = [xgb_final, rf_final, mlp_a_final, mlp_b_final]

In [ ]:
# =============================================================================
# Inference pipeline — generate test predictions
# =============================================================================

def get_base_predictions(models, X):
    """Get probability predictions from all base models."""
    preds = np.zeros((X.shape[0], len(models)))
    # Tree-based models
    preds[:, 0] = models[0].predict_proba(X)[:, 1]       # XGBoost
    preds[:, 1] = models[1].predict_proba(X)[:, 1]       # RandomForest
    # NN models
    preds[:, 2] = predict_mlp(models[2], X)               # MLP-A
    preds[:, 3] = predict_mlp(models[3], X)               # MLP-B
    return preds

# Generate test predictions
test_base_preds = get_base_predictions(final_models, X_test)

# --- 1. Individual model predictions ---
individual_preds = {name: test_base_preds[:, i] for i, name in enumerate(base_model_names)}

# --- 2. Simple averaging ---
avg_preds = test_base_preds.mean(axis=1)

# --- 3. Stacking ensemble ---
stacked_preds = meta_model.predict_proba(test_base_preds)[:, 1]

print("Test predictions ready.")

In [ ]:
# =============================================================================
# Evaluation — compare all approaches
# =============================================================================

results = []

# Individual models
for name in base_model_names:
    auc = roc_auc_score(y_test, individual_preds[name])
    results.append({'Model': name, 'Type': 'Individual', 'AUC-ROC': auc})

# Simple average
avg_auc = roc_auc_score(y_test, avg_preds)
results.append({'Model': 'Simple Average', 'Type': 'Ensemble', 'AUC-ROC': avg_auc})

# Stacking
stack_auc = roc_auc_score(y_test, stacked_preds)
results.append({'Model': 'Stacking (LogReg Meta)', 'Type': 'Ensemble', 'AUC-ROC': stack_auc})

results_df = pd.DataFrame(results).sort_values('AUC-ROC', ascending=False)

print("\n" + "="*55)
print("TEST SET RESULTS — sorted by AUC-ROC")
print("="*55)
for _, row in results_df.iterrows():
    print(f"  {row['Model']:30s}  {row['AUC-ROC']:.4f}")

# Improvement over best individual
best_individual_auc = results_df[results_df['Type'] == 'Individual']['AUC-ROC'].max()
improvement = stack_auc - best_individual_auc
print(f"\nStacking improvement over best individual: {improvement:+.4f}")
print(f"Stacking improvement over simple average: {stack_auc - avg_auc:+.4f}")

In [ ]:
# --- Barplot comparison ---
fig, ax = plt.subplots(figsize=(10, 5))

colors = ['#3498db' if t == 'Individual' else '#e74c3c' for t in results_df['Type']]
bars = ax.barh(results_df['Model'], results_df['AUC-ROC'], color=colors)

# Annotate bars
for bar, val in zip(bars, results_df['AUC-ROC']):
    ax.text(bar.get_width() + 0.002, bar.get_y() + bar.get_height()/2,
            f'{val:.4f}', va='center', fontsize=11)

ax.set_xlabel('AUC-ROC')
ax.set_title('Test Set AUC-ROC: Individual Models vs Ensembles')
ax.axvline(x=best_individual_auc, color='#3498db', linestyle='--', alpha=0.5,
           label=f'Best Individual = {best_individual_auc:.4f}')
ax.legend(loc='lower right')
ax.set_xlim(0.65, max(results_df['AUC-ROC']) + 0.03)
plt.tight_layout()
plt.show()

In [ ]:
# --- OOF correlation heatmap — are base models diverse? ---
oof_df = pd.DataFrame(oof_predictions, columns=base_model_names)

fig, ax = plt.subplots(figsize=(7, 5))
sns.heatmap(oof_df.corr(), annot=True, fmt='.3f', cmap='coolwarm',
            vmin=0, vmax=1, ax=ax)
ax.set_title('OOF Prediction Correlation — Base Model Diversity')
plt.tight_layout()
plt.show()

In [ ]:
# --- Meta-model weight plot ---
fig, ax = plt.subplots(figsize=(8, 4))

bars = ax.barh(coef_df['Model'], coef_df['Weight'],
               color=['#2ecc71' if w > 0 else '#e74c3c' for w in coef_df['Weight']])
ax.axvline(x=0, color='black', linewidth=0.8)
ax.set_xlabel('Logistic Regression Coefficient')
ax.set_title('Meta-Model: How Much Each Base Model Contributes')

for bar, val in zip(bars, coef_df['Weight']):
    x_pos = bar.get_width() + (0.02 if val >= 0 else -0.08)
    ax.text(x_pos, bar.get_y() + bar.get_height()/2,
            f'{val:.3f}', va='center', fontsize=11)

plt.tight_layout()
plt.show()

In [ ]:
# =============================================================================
# Comprehensive metrics — beyond AUC-ROC
# =============================================================================
from sklearn.metrics import (
    precision_score, recall_score, f1_score, brier_score_loss,
    precision_recall_curve, average_precision_score, RocCurveDisplay
)

# Hard predictions at threshold = 0.5
stacked_hard = (stacked_preds >= 0.5).astype(int)

# Per-model metrics
metrics_rows = []
for name in base_model_names:
    preds_hard = (individual_preds[name] >= 0.5).astype(int)
    metrics_rows.append({
        'Model': name,
        'Precision': precision_score(y_test, preds_hard),
        'Recall':    recall_score(y_test, preds_hard),
        'F1':        f1_score(y_test, preds_hard),
        'Brier':     brier_score_loss(y_test, individual_preds[name]),
        'Avg Prec':  average_precision_score(y_test, individual_preds[name]),
    })

# Ensembles
for label, preds in [('Simple Average', avg_preds), ('Stacking', stacked_preds)]:
    preds_hard = (preds >= 0.5).astype(int)
    metrics_rows.append({
        'Model': label,
        'Precision': precision_score(y_test, preds_hard),
        'Recall':    recall_score(y_test, preds_hard),
        'F1':        f1_score(y_test, preds_hard),
        'Brier':     brier_score_loss(y_test, preds),
        'Avg Prec':  average_precision_score(y_test, preds),
    })

metrics_df = pd.DataFrame(metrics_rows).set_index('Model')
print("Comprehensive Test Metrics:\n")
print(metrics_df.round(4).to_string())

In [ ]:
# --- ROC Curves: all models on one plot ---
fig, ax = plt.subplots(figsize=(9, 7))

colors = ['#3498db', '#2ecc71', '#9b59b6', '#f39c12', '#e74c3c', '#1abc9c']
for i, name in enumerate(base_model_names):
    RocCurveDisplay.from_predictions(y_test, individual_preds[name],
                                     ax=ax, name=f'{name}', color=colors[i])

RocCurveDisplay.from_predictions(y_test, avg_preds,
                                 ax=ax, name='Simple Average', color=colors[4],
                                 linestyle='--', linewidth=2)
RocCurveDisplay.from_predictions(y_test, stacked_preds,
                                 ax=ax, name='Stacking', color=colors[5],
                                 linewidth=2.5)

ax.plot([0, 1], [0, 1], 'k--', alpha=0.2, label='Random')
ax.set_title('ROC Curves — All Models vs Ensembles')
ax.legend(loc='lower right', fontsize=9)
plt.tight_layout()
plt.show()

In [ ]:
# --- Precision-Recall curves (more informative than ROC for imbalanced data) ---
fig, ax = plt.subplots(figsize=(9, 7))

for i, name in enumerate(base_model_names):
    prec, rec, _ = precision_recall_curve(y_test, individual_preds[name])
    ap = average_precision_score(y_test, individual_preds[name])
    ax.plot(rec, prec, color=colors[i], label=f'{name} (AP={ap:.3f})', linewidth=1.5)

prec, rec, _ = precision_recall_curve(y_test, avg_preds)
ap_avg = average_precision_score(y_test, avg_preds)
ax.plot(rec, prec, color=colors[4], linestyle='--', linewidth=2,
        label=f'Simple Average (AP={ap_avg:.3f})')

prec, rec, _ = precision_recall_curve(y_test, stacked_preds)
ap_stack = average_precision_score(y_test, stacked_preds)
ax.plot(rec, prec, color=colors[5], linewidth=2.5,
        label=f'Stacking (AP={ap_stack:.3f})')

# Baseline: random classifier
baseline = y_test.mean()
ax.axhline(y=baseline, color='gray', linestyle=':', alpha=0.5,
           label=f'Baseline ({baseline:.2f})')

ax.set_xlabel('Recall')
ax.set_ylabel('Precision')
ax.set_title('Precision-Recall Curves — All Models vs Ensembles')
ax.legend(loc='upper right', fontsize=9)
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
plt.tight_layout()
plt.show()

In [ ]:
# --- Confusion matrix for the stacking ensemble ---
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))

# Raw counts
cm = confusion_matrix(y_test, stacked_hard)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax1,
            xticklabels=['Non-Default', 'Default'],
            yticklabels=['Non-Default', 'Default'])
ax1.set_xlabel('Predicted')
ax1.set_ylabel('Actual')
ax1.set_title(f'Confusion Matrix — Stacking Ensemble')

# Normalised by row (recall per class)
cm_norm = confusion_matrix(y_test, stacked_hard, normalize='true')
sns.heatmap(cm_norm, annot=True, fmt='.2%', cmap='Blues', ax=ax2,
            xticklabels=['Non-Default', 'Default'],
            yticklabels=['Non-Default', 'Default'])
ax2.set_xlabel('Predicted')
ax2.set_ylabel('Actual')
ax2.set_title('Normalised by Row (Recall)')

plt.tight_layout()
plt.show()

In [ ]:
# --- Threshold analysis: how precision/recall trade off ---
thresholds = np.linspace(0.1, 0.9, 50)
precisions, recalls, f1s = [], [], []

for t in thresholds:
    hard = (stacked_preds >= t).astype(int)
    precisions.append(precision_score(y_test, hard, zero_division=0))
    recalls.append(recall_score(y_test, hard, zero_division=0))
    f1s.append(f1_score(y_test, hard, zero_division=0))

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(thresholds, precisions, label='Precision', color='#3498db', linewidth=2)
ax.plot(thresholds, recalls,    label='Recall',    color='#e74c3c', linewidth=2)
ax.plot(thresholds, f1s,        label='F1 Score',  color='#2ecc71', linewidth=2)

# Find optimal F1 threshold
best_idx = np.argmax(f1s)
ax.axvline(x=thresholds[best_idx], color='gray', linestyle='--', alpha=0.5,
           label=f'Best F1 @ {thresholds[best_idx]:.2f} (F1={f1s[best_idx]:.3f})')

ax.set_xlabel('Threshold')
ax.set_ylabel('Score')
ax.set_title('Stacking Ensemble — Precision / Recall / F1 vs. Threshold')
ax.legend()
ax.set_ylim(0, 1)
plt.tight_layout()
plt.show()

In [ ]:
# --- Save models ---
import joblib

# Tree-based models
joblib.dump(xgb_final, f"{OUT_DIR}/xgb_final.pkl")
joblib.dump(rf_final,  f"{OUT_DIR}/rf_final.pkl")
joblib.dump(meta_model, f"{OUT_DIR}/meta_model.pkl")

# NN models
torch.save(mlp_a_final.state_dict(), f"{OUT_DIR}/mlp_a_final.pt")
torch.save(mlp_b_final.state_dict(), f"{OUT_DIR}/mlp_b_final.pt")

# Model config
config = {
    'base_model_names': base_model_names,
    'input_dim': input_dim,
    'n_folds': N_FOLDS,
}
with open(f"{OUT_DIR}/config.pkl", 'wb') as f:
    pickle.dump(config, f)

# Results summary
results_df.to_csv(f"{OUT_DIR}/results.csv", index=False)
metrics_df.to_csv(f"{OUT_DIR}/metrics.csv")

print(f"Models saved to {OUT_DIR}/")
print(f"  xgb_final.pkl, rf_final.pkl, meta_model.pkl")
print(f"  mlp_a_final.pt, mlp_b_final.pt")
print(f"  config.pkl, results.csv, metrics.csv")

# --- Download artifacts in Colab ---
if IN_COLAB:
    from google.colab import files
    import shutil
    shutil.make_archive(f"{OUT_DIR}/models", 'zip', OUT_DIR)
    print(f"\nDownloading models.zip...")
    files.download(f"{OUT_DIR}/models.zip")